# AutoData — Colab GPU research & backend benchmark

This notebook runs the **same backend modules used by the project**, on a Colab CUDA GPU. It is designed around the current research question:

> **Why does E0/raw data sometimes outperform E1/E2 (AI-ready) data?**

The notebook does four things:
1. loads the project robustly from an uploaded ZIP or an already-extracted folder;
2. builds E0/E1/E2 from one shared split/sample;
3. runs **representation diagnostics + feature-group ablations**;
4. runs multi-seed GPU training and exports results as CSV/JSON.

The default metric is **PR-AUC**, because fraud is highly imbalanced and accuracy can be misleading.

Research rationale: FT-Transformer treats feature representation as part of the architecture, and numerical-embedding research shows that preserving continuous numeric information can materially affect tabular deep-learning performance. Therefore this notebook tests representation changes and feature groups separately instead of assuming more preprocessing is always better.


In [ ]:
# 0) Confirm a Colab GPU runtime
import os, sys, subprocess, textwrap, json, shutil, zipfile
from pathlib import Path

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
except Exception as e:
    print("Torch check failed before setup:", e)

!nvidia-smi || true

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Colab: Runtime > Change runtime type > choose T4/L4/A100, then rerun.")


In [ ]:
# 1) Locate or upload the project
# Supported paths:
#   A) /content/transaction-data-intelligence already exists
#   B) /content/v2.zip already exists
#   C) upload the project ZIP when prompted

from pathlib import Path
import os, zipfile

CONTENT = Path("/content")
PROJECT_ROOT = CONTENT / "transaction-data-intelligence"
ZIP_CANDIDATES = [CONTENT / "v2.zip", CONTENT / "transaction-data-intelligence.zip"]

if not (PROJECT_ROOT / "src").exists():
    zpath = next((p for p in ZIP_CANDIDATES if p.exists()), None)
    if zpath is None:
        from google.colab import files
        print("Upload the latest project ZIP (for example v2_updated.zip)")
        uploaded = files.upload()
        names = list(uploaded)
        if not names:
            raise RuntimeError("No ZIP uploaded")
        zpath = CONTENT / names[0]

    print("Extracting", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CONTENT)

# Some ZIPs may contain one wrapper directory. Find the actual project root deterministically.
if not (PROJECT_ROOT / "src").exists():
    matches = [p for p in CONTENT.rglob("config.yaml") if (p.parent / "src").exists()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely locate project root. Found: {matches}")
    PROJECT_ROOT = matches[0].parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())


In [ ]:
# 2) Install the backend dependencies used by this repository
# Core benchmark does NOT require transformers/peft/torchao.
# Keeping the core environment small avoids optional-package version conflicts.

!pip install -q -r requirements.txt

# Restart is usually not needed. Re-import torch and confirm CUDA.
import torch
assert torch.cuda.is_available(), "CUDA disappeared after install"
DEVICE = "cuda"
print("GPU ready:", torch.cuda.get_device_name(0))


## 3) Choose data and experiment size

For the first run, use the built-in synthetic transaction generator to verify the notebook end-to-end. Then set `DATA_PATH` to a real CSV/JSON file in Colab.

Recommended progression:
- smoke test: `ROWS = 10_000`, `SEEDS = [42]`, 3–5 epochs;
- research run: `ROWS = 50_000` or more, `SEEDS = [42,123,456]`, 10–20 epochs;
- real client data: use a temporally representative sample first, then scale upward.


In [ ]:
# User controls
DATA_PATH = None       # e.g. "/content/transactions.csv"; None => built-in synthetic data
TARGET = None          # set explicitly if auto-detection picks the wrong target, e.g. "is_fraud"
ROWS = 50_000          # use "full" only after the benchmark is stable
SEEDS = [42, 123, 456]
EPOCHS = 12
PATIENCE = 4
BATCH_SIZE = 512       # lower to 256/128 if GPU memory is tight
RUN_FULL_MATRIX = True # False = faster baseline only

print({"ROWS": ROWS, "SEEDS": SEEDS, "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE})


In [ ]:
# 4) Load data using the project's own ingestion code
from pathlib import Path
from src.ingestion.demo_data import make_transactions
from src.ingestion.loader import load_dataset
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.ingestion.schema_detector import detect_schema
from src.utils.config import ROOT, load_config

cfg = load_config()
if DATA_PATH:
    ds = load_dataset(Path(DATA_PATH), metadata_dir=ROOT / cfg["project"]["data_dir"] / "raw" / "_metadata")
    df, dataset_id = ds.df, ds.metadata.dataset_id
else:
    # Enough data to exercise temporal/history behaviour; clearly synthetic, not production evidence.
    df = make_transactions(n_cards=250, days=180, seed=0)
    dataset_id = "colab_gpu_synthetic"

schema = detect_schema(df, dataset_id)
roles = detect_roles(df, schema, target=TARGET) if TARGET else detect_roles(df, schema)
ps = schema_for_profiling(schema, roles, df)

print(f"Rows={len(df):,}, columns={len(df.columns)}")
print(f"target={roles.target!r}, task={roles.task!r}, entity={roles.entity!r}, datetime={roles.datetime!r}")
if roles.task != "binary_classification":
    raise ValueError("This fraud benchmark currently expects a binary target. Set TARGET explicitly if detection was wrong.")


In [ ]:
# 5) Build the shared split and baseline E0/E1/E2
from src.preprocessing.levels import DataPreparer, infer_roles

level_roles = infer_roles(df, ps, roles)
preparer = DataPreparer(df, ps, level_roles, cfg, rows=ROWS, seed=cfg["project"]["seed"])
split_info = preparer.prepare_split()
print(json.dumps(split_info["sample"], indent=2, default=str))

levels = {lv: preparer.build(lv) for lv in ["E0", "E1", "E2"]}
for lv, p in levels.items():
    print(f"{lv}: {len(p.features)} features | numeric={len(p.numeric)} categorical={len(p.categorical)}")


In [ ]:
# 6) Information-loss / representation diagnostics BEFORE expensive GPU training
from src.evaluation.signal_diagnostics import compare_levels, diagnose_level
from IPython.display import display

signal_table = compare_levels(levels, cfg["representation"])
display(signal_table)

for lv in ["E0", "E1", "E2"]:
    d = diagnose_level(levels[lv], cfg["representation"])
    details = sorted(d["features"], key=lambda x: (x["univariate_test_auc"] is None, -(x["univariate_test_auc"] or 0)))
    print(f"\n{lv} top diagnostic features")
    display(__import__('pandas').DataFrame(details).head(15))


### How to interpret the diagnostic table

Red flags worth investigating before blaming the model:
- high categorical `test_unknown_rate` → the vocabulary is collapsing unseen/rare values;
- very low `representation_retention` → aggressive binning is discarding numeric detail;
- many `near_constant_representations` → engineered features are adding little usable variation;
- E1/E2 lower `best/median_univariate_test_auc` than E0 → preprocessing may have removed direct signal;
- E2 has many more positions but no stronger signal → extra features may be noise for the available sample/model capacity.


In [ ]:
# 7) GPU baseline: same model/settings/seeds for E0/E1/E2
from src.evaluation.experiment import run_comparison_multiseed, aggregate_seeds

settings = {
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
}

baseline = run_comparison_multiseed(
    preparer, ["E0", "E1", "E2"], cfg, SEEDS,
    settings=settings, device=DEVICE,
    progress=lambda done,total,seed,r: print(
        f"[{done:02d}/{total}] seed={seed} {r.level}: PR-AUC={r.metrics['test']['pr_auc']} epochs={r.epochs_run}")
)
baseline_table = aggregate_seeds(baseline)
display(baseline_table)


## 8) Controlled research matrix

This isolates two hypotheses instead of changing everything at once:

**Representation hypothesis** — quantile tokens may discard magnitude/order information. Test continuous numeric embeddings while keeping the data level fixed.

**Feature-noise hypothesis** — E2 may be worse because some engineered feature families hurt. Test temporal/history/sequence independently.

Every variant reconstructs a `DataPreparer` with the same deterministic split/sample seed. The only intended change is the named configuration override.


In [ ]:
# 8a) Variant runner
import copy, pandas as pd, numpy as np
from src.evaluation.experiment import run_comparison_multiseed, aggregate_seeds


def make_preparer(config):
    p = DataPreparer(df, ps, level_roles, config, rows=ROWS, seed=cfg["project"]["seed"])
    p.prepare_split()
    return p


def run_variant(name, level, config, seeds=SEEDS):
    print("\n===", name, "===")
    p = make_preparer(config)
    out = run_comparison_multiseed(
        p, [level], config, seeds, settings=settings, device=DEVICE,
        progress=lambda done,total,seed,r: print(f"  seed={seed}: PR-AUC={r.metrics['test']['pr_auc']}")
    )
    table = aggregate_seeds(out).reset_index().rename(columns={"Level":"level"})
    table.insert(0, "variant", name)
    return out, table

variants = []

# Baseline rows from the already-completed run
base_table = baseline_table.reset_index().rename(columns={"Level":"level"})
base_table.insert(0, "variant", "baseline_quantile")
variants.append(base_table)

if RUN_FULL_MATRIX:
    # A. E1 continuous numeric representation
    c = copy.deepcopy(cfg)
    c["representation"]["numeric_mode"] = "continuous"
    c["representation"]["numeric_coarse_bins"] = 0
    _, t = run_variant("E1_continuous", "E1", c)
    variants.append(t)

    # B. E2 continuous numeric representation
    c = copy.deepcopy(cfg)
    c["representation"]["numeric_mode"] = "continuous"
    c["representation"]["numeric_coarse_bins"] = 0
    _, t = run_variant("E2_continuous", "E2", c)
    variants.append(t)

    # C. E2 feature-family ablations under the current conservative quantile representation
    for group in [["temporal"], ["history"], ["sequence"], ["temporal","history"], ["history","sequence"]]:
        c = copy.deepcopy(cfg)
        c["features"]["enabled_groups"] = group
        name = "E2_" + "+".join(group)
        _, t = run_variant(name, "E2", c)
        variants.append(t)

research_table = pd.concat(variants, ignore_index=True)
display(research_table)


In [ ]:
# 9) Rank by mean PR-AUC for investigation (NOT a production claim)
# We use this only to identify which experiments deserve follow-up; require 3 seeds and inspect std.
cols = [c for c in research_table.columns if "pr_auc" in c.lower() or c in ["variant","level","Seeds"]]
display(research_table[cols].sort_values("pr_auc mean", ascending=False))


In [ ]:
# 10) Save reproducible GPU results
from datetime import datetime
from pathlib import Path

out_dir = PROJECT_ROOT / "experiments" / "colab_gpu"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = out_dir / f"research_matrix_{stamp}.csv"
json_path = out_dir / f"research_context_{stamp}.json"
research_table.to_csv(csv_path, index=False)

context = {
    "dataset_id": dataset_id,
    "rows_requested": ROWS,
    "seeds": SEEDS,
    "device": torch.cuda.get_device_name(0),
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "split": split_info,
    "representation_default": cfg["representation"],
    "feature_default": cfg["features"],
}
json_path.write_text(json.dumps(context, indent=2, default=str))
print(csv_path)
print(json_path)

# Download both files to your computer.
from google.colab import files
files.download(str(csv_path))
files.download(str(json_path))


## 11) Decision rules after the run

Do **not** change a production default because one seed or one subset wins.

Use the following rule:
- require at least 3 seeds;
- use PR-AUC as the primary fraud metric;
- inspect variance as well as mean;
- prefer a simpler transformation when performance is statistically indistinguishable;
- if raw still wins, inspect the diagnostics and feature ablations before adding model complexity;
- validate the chosen setup on a second dataset/source before making it an AutoData default.

Likely outcomes:
- **E1 continuous > E1 quantile** → magnitude loss is real; promote continuous representation for suitable numeric features.
- **E2 sequence/history > E2 full** → some engineered groups are noise; make feature-family selection data-dependent.
- **E0 remains best across seeds** → preserve more raw signal and redesign the transformation policy rather than adding more preprocessing.

The project should eventually treat preprocessing as a **measured policy selection problem**, not a fixed ladder where E2 is assumed to be superior.
